# Pattern A · On-behalf-of token exchange (RFC 8693)

An agent acts for the signed-in user. Each authorization server **trusts the one before it**, and each hop is one token exchange:

| Step | Who asks | Okta app | Gets | From | Grant |
|---|---|---|---|---|---|
| 1 | Ada (sign-in) | `anyco-eng-login-app` | T1 | `anyco-users-as` | authorization code + PKCE |
| 2 | your backend | `anyco-console-backend-app` | T2 | `anyco-agent-a-as` (trusts `anyco-users-as`) | token exchange, subject T1 |
| 3 | the agent | `anyco-pr-review-agent-app` | T3 | `anyco-gateway-as` (trusts `anyco-agent-a-as`) | token exchange, subject T2 |

No agent and no app run here. Step 1 uses `requests`; steps 2 and 3 use AgentCore Identity's `@requires_access_token` decorator, the same one the agent uses on AgentCore Runtime.

**Before you run it:** Okta is set up (`docs/okta-setup.md`), `okta_values.py` and `deploy.py` have run (they create the agent's provider, `anyco-pr-review-agent-obo`), and `.env` has `DEMO_USER_PASSWORD`. This notebook creates a workload identity (`anyco-notebook-workload`) and one provider for the backend (`anyco-notebook-backend-exchange`); the last cell removes them.

In [ ]:
import asyncio, base64, hashlib, json, os, secrets, sys, time, urllib.parse, uuid
sys.path.insert(0, "..")                       # the repo root: config.py reads ../.env and ../state.json
import requests
import config as C

os.environ["AWS_REGION"] = C.REGION            # the decorator builds its own client from AWS_REGION
if C.AWS_PROFILE:
    os.environ["AWS_PROFILE"] = C.AWS_PROFILE
from bedrock_agentcore.identity.auth import requires_access_token
from bedrock_agentcore.runtime.context import BedrockAgentCoreContext
from bedrock_agentcore.services.identity import IdentityClient

E, st = C.os.environ, C.state()
ac = C.session().client("bedrock-agentcore-control")   # to create the notebook's providers
identity = IdentityClient(C.REGION)                     # to bind a JWT to the workload identity

# An older boto3 doesn't know the KMS-signed client assertion (privateKeyJwtConfig) the providers use.
_fields = ac.meta.service_model.shape_for("CustomOauth2ProviderConfigInput").members
if "privateKeyJwtConfig" not in _fields:
    import boto3
    raise RuntimeError(f"This kernel's boto3 {boto3.__version__} is too old ({sys.executable}). "
                       "Run the notebook on the repo's .venv kernel, or: pip install -r notebooks/requirements.txt")

def claims(jwt):
    return json.loads(base64.urlsafe_b64decode(jwt.split(".")[1] + "=="))

def show(name, jwt):
    """A bearer token: the raw value shortened, then the header and every claim. The signature is shown
    only as a length, so nothing printed here is a usable credential."""
    h, p, sig = jwt.split(".")
    dec = lambda x: json.loads(base64.urlsafe_b64decode(x + "=="))
    print(f"{name}  ·  {len(jwt)} chars  ·  {jwt[:16]}…{jwt[-6:]}")
    print(json.dumps({"header": dec(h), "payload": dec(p), "signature": f"<{len(sig)} chars, not shown>"}, indent=1))

def act_chain(jwt):
    """Every actor in the token, newest first."""
    out, a = [], claims(jwt).get("act")
    while isinstance(a, dict):
        out.append(f"{a['sub']} ({a.get('sub_profile', '')})"); a = a.get("act")
    return out

WORKLOAD = "anyco-notebook-workload"           # a standalone workload identity, to bind each subject token
try:
    ac.create_workload_identity(name=WORKLOAD, tags={"project": "anyco-agentid"})
except ac.exceptions.ValidationException as e:
    if "already exists" not in str(e): raise

def bind(jwt):
    """Make this JWT the subject of the next decorated call: AgentCore Identity returns a workload access
    token bound to it, and the decorators read the one in the context."""
    wat = identity.get_workload_access_token(WORKLOAD, user_token=jwt)["workloadAccessToken"]
    BedrockAgentCoreContext.set_workload_access_token(wat)

def provider(name, cfg):
    """Create or update an AgentCore Identity OAuth2 provider."""
    try:
        ac.create_oauth2_credential_provider(name=name, credentialProviderVendor="CustomOauth2",
                                             oauth2ProviderConfigInput=cfg, tags={"project": "anyco-agentid"})
    except (ac.exceptions.ConflictException, ac.exceptions.ValidationException) as e:
        if "already exists" not in str(e) and not isinstance(e, ac.exceptions.ConflictException): raise
        ac.update_oauth2_credential_provider(name=name, credentialProviderVendor="CustomOauth2", oauth2ProviderConfigInput=cfg)

ACCESS_TOKEN = "urn:ietf:params:oauth:token-type:access_token"
ID_JAG = "urn:ietf:params:oauth:token-type:id-jag"
print("ready ·", C.OKTA_DOMAIN, "·", C.REGION)

## 1 · Ada signs in: T1
T1's audience is `api://anyco-pr-agent-aud`. No agent accepts it; it stays with your backend.

In [ ]:
# Ada signs in to anyco-eng-login-app at anyco-users-as: authorization code + PKCE, with plain requests.
# (A browser would show Okta's sign-in page; here Okta's authn API gives a session token instead.)
ok = st["okta"]
s = requests.Session()
authn = s.post(f"{C.OKTA_DOMAIN}/api/v1/authn", timeout=20,
               json={"username": "ada.lovelace@example.com", "password": E["DEMO_USER_PASSWORD"]}).json()
verifier = secrets.token_urlsafe(48)
challenge = base64.urlsafe_b64encode(hashlib.sha256(verifier.encode()).digest()).rstrip(b"=").decode()
r = s.get(f"{ok['users_issuer']}/v1/authorize", allow_redirects=False, timeout=20, params={
    "client_id": ok["login_client_id"], "response_type": "code", "scope": "openid profile email pr-agent.invoke",
    "redirect_uri": C.REDIRECT_URI, "state": uuid.uuid4().hex, "nonce": uuid.uuid4().hex,
    "code_challenge": challenge, "code_challenge_method": "S256", "sessionToken": authn["sessionToken"]})
code = urllib.parse.parse_qs(urllib.parse.urlparse(r.headers["Location"]).query)["code"][0]
tokens = s.post(f"{ok['users_issuer']}/v1/token", auth=(ok["login_client_id"], E["ANYCO_LOGIN_CLIENT_SECRET"]), timeout=20,
                data={"grant_type": "authorization_code", "code": code, "redirect_uri": C.REDIRECT_URI,
                      "code_verifier": verifier}).json()
t1 = tokens["access_token"]
show("T1 · Ada's sign-in token", t1)

## 2 · Your backend: T1 → T2 (token exchange at `anyco-agent-a-as`)
An AgentCore Identity provider holds the backend's Okta app (client ID and secret) and points at `anyco-agent-a-as`. The decorator asks it for a token **on behalf of** the JWT bound to the workload identity: here, T1.

In [ ]:
BACKEND = "anyco-notebook-backend-exchange"
provider(BACKEND, {"customOauth2ProviderConfig": {
    "oauthDiscovery": {"discoveryUrl": f"{ok['agent_a_issuer']}/.well-known/openid-configuration"},
    "clientId": ok["backend_client_id"], "clientSecret": E["ANYCO_BACKEND_CLIENT_SECRET"],   # anyco-console-backend-app
    "clientAuthenticationMethod": "CLIENT_SECRET_BASIC",
    "onBehalfOfTokenExchangeConfig": {"grantType": "TOKEN_EXCHANGE",
                                      "tokenExchangeGrantTypeConfig": {"actorTokenContent": "NONE"}}}})

@requires_access_token(provider_name=BACKEND, scopes=["agent-a.invoke"], auth_flow="ON_BEHALF_OF_TOKEN_EXCHANGE",
                       force_authentication=True,
                       custom_parameters={"audience": ok["agent_a_audience"], "subject_token_type": ACCESS_TOKEN})
async def get_t2(*, access_token):
    return access_token

bind(t1)                  # the subject of this exchange is T1
t2 = await get_t2()
show("T2 · for the agent", t2)
print("act:", act_chain(t2))

## 3 · The agent: T2 → T3 (token exchange at `anyco-gateway-as`)
The same decorator the agent runs on AgentCore Runtime, with the provider `deploy.py` created for the agent's Okta app. `anyco-gateway-as` accepts T2 because it trusts `anyco-agent-a-as`.

In [ ]:
@requires_access_token(provider_name=C.N["obo_provider"], scopes=["pr:read", "pr:review"],
                       auth_flow="ON_BEHALF_OF_TOKEN_EXCHANGE", force_authentication=True,
                       custom_parameters={"audience": ok["gateway_audience"], "subject_token_type": ACCESS_TOKEN})
async def get_t3(*, access_token):
    return access_token

bind(t2)                  # the subject is now T2
t3 = await get_t3()
show("T3 · for the Gateway", t3)

## 4 · The trail
Okta adds the requesting client as the outermost `act` at each exchange and keeps the earlier ones.

In [ ]:
for name, tok in (("T1", t1), ("T2", t2), ("T3", t3)):
    c = claims(tok)
    print(f"{name}: iss …{c['iss'][-20:]} · aud {c['aud']} · sub {c['sub']} · cid {c['cid']}")
    for i, a in enumerate(act_chain(tok)):
        print(f"      {'act' + '.act' * i}.sub = {a}")

## 5 · Why each hop needs its own token
`anyco-gateway-as` trusts only `anyco-agent-a-as`, so exchanging T1 there directly is refused.

In [ ]:
r = requests.post(f"{ok['gateway_issuer']}/v1/token", auth=(ok["agent_client_id"], E["OKTA_AGENT_CLIENT_SECRET"]), timeout=20,
                  data={"grant_type": "urn:ietf:params:oauth:grant-type:token-exchange", "subject_token": t1,
                        "subject_token_type": ACCESS_TOKEN, "audience": ok["gateway_audience"], "scope": "pr:read"})
print(r.status_code, r.json().get("error"), "·", r.json().get("error_description"))

## Clean up
Removes the workload identity and the provider this notebook created. The deployed demo is untouched.

In [ ]:
NOTEBOOK_PROVIDERS = ["anyco-notebook-backend-exchange"]

for name in NOTEBOOK_PROVIDERS:
    try:
        ac.delete_oauth2_credential_provider(name=name); print("deleted provider", name)
    except Exception as e:
        print("skip", name, str(e)[:80])
try:
    ac.delete_workload_identity(name=WORKLOAD); print("deleted workload identity", WORKLOAD)
except Exception as e:
    print("skip", WORKLOAD, str(e)[:80])